# Chapter 17 companion — Benders on EX-3

The expansion problem end to end on an instance small enough to enumerate:

1. the network and the four plans, with the **truth** computed by enumeration;
2. the Benders loop and its bound ladder;
3. the **feasibility** cut, and the soft cut that replaces it;
4. the **minorant test** applied to every cut the run produces.

The point of an instance with four plans is that every claim the algorithm makes can be
*checked* rather than believed.

Read Chapter 17 in the separately supplied Part V volume. Follow the
[Part V setup guide](../../PART-V-COMPANIONS.md), select its Julia project/kernel,
and restart a separate kernel for this notebook before running all cells.

Repository-only source/numerical audit: `tools/verify/verify_ch17_companion_review2.py`.
It independently assembles the LP; it is not notebook execution or live HTML acceptance.
The historical `verify_ch17.py` is retired because its capacity-only derivative is obsolete.

Normalization: each scenario is one hour; investment300/150 are comparable
period-equivalent charges, not annual capital budgets. Costs8/12/40/500 are
dollars per MWh. An annual study needs representative-hour weights and an
explicit annuity/lifetime assumption before adding annualized capital.


In [1]:
using JuMP, HiGHS, Printf, Statistics

const C_A, CAP_A = 8.0, 100.0        # bus A; A-C and B-C have equal reactance
const C_B, CAP_B = 12.0, 90.0        # bus B; no electrical-distance asymmetry
const C_PK, CAP_PK = 40.0, 60.0      # candidate peaker at bus C
const CDEF = 500.0                   # unserved energy
const LIM = 60.0                     # every existing line
const COST_Z = [300.0, 150.0]        # period-equivalent charges: line, peaker; one-hour scenarios
const DEMANDS = [120.0, 130.0]
const PROB = [0.5, 0.5]
const BIGM_VALID = 60.0
const PLANS = [(0, 0), (1, 0), (0, 1), (1, 1)]

"""Only a proved infeasible solve is a feasibility result; other failures are errors."""
function operating_outcome(status, primal, dual)
    status == MOI.OPTIMAL && return :optimal
    status == MOI.INFEASIBLE && return :infeasible
    error("Operating solve not certified: termination=$status, primal=$primal, dual=$dual. No feasibility conclusion or cut is available.")
end

"""One operating scenario on the triangle. Bus C is slack and every reactance is 1, so
f_AC = thA, f_BC = thB, and Kirchhoff's voltage law is the row fAB = thA - thB. The
candidate is a second A--C line carrying fk with a disjunctive voltage law.

kvl = :full     the DC model -- the loop law and the candidate's disjunctive row
      :project  the two-oracle relaxation -- existing physics kept, candidate's row dropped
      :none     transportation -- capacities only, no voltage law anywhere
"""
function operate(z, d; kvl = :full, bigM = BIGM_VALID, fixvar = nothing)
    z1, z2 = z
    m = Model(HiGHS.Optimizer); set_silent(m)
    @variable(m, 0 <= gA <= CAP_A); @variable(m, 0 <= gB <= CAP_B)
    @variable(m, gPK >= 0); @variable(m, df >= 0)
    @constraint(m, cap_pk, gPK <= CAP_PK * z2)
    @variable(m, thA); @variable(m, thB); @variable(m, fk); @variable(m, fAB)

    @constraint(m, kclA, gA - thA - fk - fAB == 0)      # KCL at A
    @constraint(m, kclB, gB + fAB - thB == 0)           # KCL at B
    @constraint(m, kclC, gPK + df + thA + thB + fk == d)# KCL at C
    if kvl in (:full, :project)
        @constraint(m, loop, fAB - thA + thB == 0)      # KVL on the existing triangle
    end
    if kvl == :full
        slack = bigM * (1 - z1)                        # the candidate's disjunction
        @constraint(m, voltage_hi, thA - fk <= slack)
        @constraint(m, voltage_lo, -(thA - fk) <= slack)
    end
    @constraint(m, -LIM <= thA <= LIM); @constraint(m, -LIM <= thB <= LIM)
    @constraint(m, -LIM <= fAB <= LIM)
    @constraint(m, capk_hi, fk <= LIM * z1); @constraint(m, capk_lo, -fk <= LIM * z1)
    if fixvar !== nothing
        for (v, val) in fixvar
            v == :gB && fix(gB, val; force = true)
            v == :df && fix(df, val; force = true)
        end
    end
    @objective(m, Min, C_A * gA + C_B * gB + C_PK * gPK + CDEF * df)
    optimize!(m)
    outcome = operating_outcome(termination_status(m), primal_status(m), dual_status(m))
    outcome == :infeasible && return nothing
    @assert primal_status(m) == MOI.FEASIBLE_POINT
    @assert dual_status(m) == MOI.FEASIBLE_POINT
    residuals = primal_feasibility_report(m)
    @assert maximum(values(residuals); init=0.0) <= 1e-6
    @assert isapprox(objective_value(m), dual_objective_value(m); atol=1e-6, rtol=1e-9)
    voltage_duals = kvl == :full ? (dual(m[:voltage_hi]), dual(m[:voltage_lo])) : (0.0,0.0)
    capacity_slope = LIM * (dual(capk_hi) + dual(capk_lo))
    voltage_slope = -bigM * sum(voltage_duals)
    (obj = objective_value(m), gA = value(gA), gB = value(gB), gPK = value(gPK),
     df = value(df), fk = value(fk), thA = value(thA), thB = value(thB), fAB = value(fAB),
     capacity_slope = capacity_slope, voltage_duals = voltage_duals,
     voltage_slope = voltage_slope,
     grad = [capacity_slope + voltage_slope, CAP_PK * dual(cap_pk)],
     priceC = dual(kclC))
end

"Expected operating cost over the two scenarios, with a subgradient in the plan."
function recourse(z; kw...)
    rows = [operate(z, d; kw...) for d in DEMANDS]
    any(isnothing, rows) && return nothing
    (value = sum(p*r.obj for (p,r) in zip(PROB,rows)),
     grad = sum(p*r.grad for (p,r) in zip(PROB,rows)),
     capacity_only = sum(p*r.capacity_slope for (p,r) in zip(PROB,rows)),
     scenarios = rows)
end

total(z; kw...) = (r = recourse(z; kw...); r === nothing ? nothing :
                   COST_Z[1] * z[1] + COST_Z[2] * z[2] + r.value)

"The master: with two binaries, enumerate. theta >= 0 is a valid floor."
function master(cuts; costs = COST_Z)
    best = nothing; arg = nothing
    for z in PLANS
        th = maximum(vcat(0.0, [b + p[1]*z[1] + p[2]*z[2] for (b, p) in cuts]))
        v = costs[1]*z[1] + costs[2]*z[2] + th
        if best === nothing || v < best - 1e-9
            best = v; arg = z
        end
    end
    best, arg
end

"Run an exactly enumerated, append-only master. Trial cost is not the incumbent.
Fixed trace mode continues after a status is reached; it is not extra required work.
Counts include actual scenario LP calls, no caching, and no diagnostic enumeration."
function benders(; oracle = (;), iters = 20, pool = [], trace = false,
                   costs = COST_Z, atol = 1e-6, rtol = 1e-9, pool_valid = true)
    cuts = copy(pool); lad = []; incumbent = Inf; incumbent_plan = nothing
    exact_calls = 0; oracle_calls = 0; evaluation_calls = 0
    known_valid = pool_valid && get(oracle, :bigM, BIGM_VALID) >= BIGM_VALID &&
                  get(oracle, :fixvar, nothing) === nothing
    for it in 1:iters
        lb, z = master(cuts; costs = costs)
        r = recourse(z; oracle...); oracle_calls += length(DEMANDS)
        r === nothing && error("Infeasible oracle: an optimality cut is unavailable; use feasibility logic.")
        if isempty(oracle)
            truth = r; exact_calls += length(DEMANDS)
        else
            truth = recourse(z); evaluation_calls += length(DEMANDS)
        end
        truth === nothing && error("Trial is not a feasible true-model incumbent")
        trial = sum(costs .* collect(z)) + truth.value
        if trial < incumbent - atol
            incumbent = trial; incumbent_plan = z
        end
        threshold = atol + rtol * max(1.0, abs(incumbent))
        oracle_total = sum(costs .* collect(z)) + r.value
        gap = incumbent - lb
        status = gap < -threshold ? "inconsistent bounds" :
                 known_valid && gap <= threshold ? "certified" :
                 abs(oracle_total - lb) <= threshold ? "oracle-model optimum" :
                 it == iters ? "iteration limit" : "running"
        push!(lad, (it = it, z = z, lb = lb, trial = trial, ub = incumbent,
                    incumbent_plan = incumbent_plan, gap = gap, status = status,
                    bound_certified = known_valid, oracle_lp = oracle_calls,
                    exact_lp = exact_calls, evaluation_lp = evaluation_calls))
        push!(cuts, (r.value - sum(r.grad .* collect(z)), Tuple(r.grad)))
        !trace && status != "running" && break
    end
    lad, cuts
end

"The minorant test: a cut is valid iff it is below R at EVERY plan."
cutval(c, z) = c[1] + c[2][1]*z[1] + c[2][2]*z[2]
isvalid(c, R; tol = 1e-6) = all(cutval(c, z) <= R[z] + tol for z in PLANS)

println("the three-bus canon of Chapter 13, reproduced:")
r1 = operate((0, 0), 120.0); r2 = operate((0, 0), 120.0; kvl = :none)
@printf("  DC model        cost %6.0f   gA %5.1f  (the loop forces the retreat)\n", r1.obj, r1.gA)
@printf("  transportation  cost %6.0f   gA %5.1f  (no loop, so A reaches the load)\n", r2.obj, r2.gA)
@printf("  the loop costs  %6.0f\n", r1.obj - r2.obj)

the three-bus canon of Chapter 13, reproduced:


  DC model        cost   1200   gA  60.0  (the loop forces the retreat)
  transportation  cost   1040   gA 100.0  (no loop, so A reaches the load)
  the loop costs     160


That 160 is the whole subject of `17-two-oracle.ipynb`. First, the truth.

In [2]:
R = Dict(z => recourse(z).value for z in PLANS)
println("plan      investment   operation      total")
for z in PLANS
    @printf("%-9s %10.0f %12.1f %10.1f\n",
            string(z), COST_Z[1]*z[1] + COST_Z[2]*z[2], R[z], total(z))
end
@assert all(isapprox(R[z],v;atol=1e-6) for (z,v) in zip(PLANS,[3700,1100,1400,1100]))
best = argmin(Dict(z => total(z) for z in PLANS))
@assert best == (1,0) && isapprox(total(best),1400;atol=1e-6)
@printf("\noptimum: %s at %.0f\n", string(best), total(best))

plan      investment   operation      total


(0, 0)             0       3700.0     3700.0


(1, 0)           300       1100.0     1400.0
(0, 1)           150       1400.0     1550.0


(1, 1)           450       1100.0     1550.0

optimum: (1, 0) at 1400


The line wins globally. The peaker and both-builds tie at1550 in expectation.
The risk dial separates **that pair**, but the line alone still dominates both
globally at the base charges; see “Risk-averse expansion” (§17.8). The widget's
separate line-charge500 comparison creates an actual peaker/line switch.

## The Benders loop


In [3]:
lad, cuts = benders()
println("it   plan      lower  incumbent       gap")
for e in lad
    @printf("%2d   %-8s %8.1f %10.1f %9.1f\n", e.it, string(e.z), e.lb, e.ub, e.ub - e.lb)
end
println("\nlower bound monotone: ", all(lad[i].lb <= lad[i+1].lb + 1e-9 for i in 1:length(lad)-1))
println("closes on the enumerated optimum: ", abs(lad[end].lb - total(best)) < 1e-6)
println("termination: ", lad[end].status)
@assert lad[end].status == "certified"
@assert isapprox(lad[end].lb,total(best);atol=1e-6)
@assert all(lad[i].lb <= lad[i+1].lb + 1e-6 for i in 1:length(lad)-1)
@assert all(lad[i+1].ub <= lad[i].ub + 1e-6 for i in 1:length(lad)-1)

it   plan      lower  incumbent       gap


 1   (0, 0)        0.0     3700.0    3700.0


 2   (0, 1)      150.0     1550.0    1400.0
 3   (1, 0)      500.0     1400.0     900.0
 4   (1, 0)     1400.0     1400.0       0.0

lower bound monotone: true
closes on the enumerated optimum: true


termination: certified


With exact master solves and an append-only valid pool, the current lower bound is
nondecreasing. Deleting cuts can lower it; keep a separate best certified LB if needed.
The **trial cost** can move either way; `ub` now stores the best exactly evaluated
feasible cost and is nonincreasing. The stopping predicate uses absolute plus relative
tolerances; a fixed-length teaching trace is available as `trace=true`, not required
confirmation iterations. All costs and bounds use the same two-scenario, one-hour model.

## Feasibility: what happens with no shortfall variable

In [4]:
println("with df fixed at 0 -- no unserved energy permitted:")
feas = Tuple{Int,Int}[]
for z in PLANS
    per = [operate(z, d; fixvar = [(:df, 0.0)]) for d in DEMANDS]
    if any(x -> x === nothing, per)
        @printf("  z=%s  INFEASIBLE\n", string(z))
    else
        push!(feas, z)
        @printf("  z=%s  feasible, expected cost %8.1f\n", string(z),
                sum(p*x.obj for (p, x) in zip(PROB, per)))
    end
end
println("\nfeasible plans: ", feas)
println("so the feasibility cut is exactly  z1 + z2 >= 1")
println("...which carries NO cost information: it forbids, it does not price.")
@assert Set(feas) == Set([(1,0),(0,1),(1,1)])


with df fixed at 0 -- no unserved energy permitted:


  z=(0, 0)  INFEASIBLE
  z=(1, 0)  feasible, expected cost   1100.0
  z=(0, 1)  feasible, expected cost   1400.0


  z=(1, 1)  feasible, expected cost   1100.0



feasible plans: [(1, 0), (0, 1), (1, 1)]
so the feasibility cut is exactly  z1 + z2 >= 1
...which carries NO cost information: it forbids, it does not price.


The soft formulation prices the shortfall instead, so the subproblem is never
infeasible and the run gets an **optimality** cut with a gradient. That is what the engine
above does by default (`df >= 0` at \$500), and it is why `R[(0,0)]` is 3700 rather than
undefined.

Softening changes the economic model; it does not reproduce hard adequacy.
Price × **physical residual** is a possible local objective-error scale, not permanent
bound noise or an inevitable gap. With a 1e-6 MWh shortage residual, 500×1e-6=0.0005
dollars and 1e7×1e-6=10 dollars. The latter is about 0.714% of1400; its acceptability
depends on the application's error budget, not a universal “worth chasing” threshold.
A solver's scaled tolerance need not be that physical residual. Audit both before
changing tolerances; do not change physically justified shortage prices for numerical
comfort. For a one-dollar local budget, the respective residual ceilings are 1/500
and 1/1e7 MWh.

## The minorant test

In [5]:
println("every cut the honest run produced, tested at all four plans:")
for (i, c) in enumerate(cuts)
    slack = [R[z] - cutval(c, z) for z in PLANS]
    @printf("  cut %d  beta %9.1f  pi (%9.1f, %9.1f)  valid: %-5s  min slack %9.1f\n",
            i, c[1], c[2][1], c[2][2], isvalid(c, R), minimum(slack))
end
println("\nA cut is valid iff it is below R at EVERY plan -- including the ones the run")
println("never visited. With four plans that test is exhaustive, so this is a proof.")
@assert all(isvalid(c,R) for c in cuts)


every cut the honest run produced, tested at all four plans:
  cut 1  beta    3700.0  pi ( -29520.0,  -27600.0)  valid: true   min slack       0.0


  cut 2  beta    1400.0  pi (  -1200.0,       0.0)  valid: true   min slack       0.0
  cut 3  beta    1100.0  pi (      0.0,       0.0)  valid: true   min slack       0.0
  cut 4  beta    1100.0  pi (      0.0,       0.0)  valid: true   min slack       0.0

A cut is valid iff it is below R at EVERY plan -- including the ones the run
never visited. With four plans that test is exhaustive, so this is a proof.


## Full RHS audit: where did the voltage-row duals go?

They must not disappear. JuMP's signed dual for a minimization `<=` row is
nonpositive. The two capacity RHS derivatives are +60, while BOTH voltage RHS
derivatives are -M. Thus the line-build slope is
`60*(dual(capk_hi)+dual(capk_lo)) - M*(dual(voltage_hi)+dual(voltage_lo))`.
The peaker uses its explicit upper-capacity row, not a clamped reduced cost at
coincident bounds. These are derivatives per unit of the dimensionless build z,
not per MW. A line build also activates physics; its full slope need not obey
the sign rule for pure capacity expansion.

At fractional z=(5/60,0), R=2450. Keeping only the capacity term gives -30240,
so the false intercept is 2450-(-30240)*(5/60)=4970: it overprices R(0,0)=3700
by 1270. Including the signed voltage contribution +15240 gives -15000 and
intercept3700. Tightness at birth alone cannot detect the wrong slope: both
intercepts were constructed to be tight there. We also check primal and dual
solution status, objective agreement, physical primal residuals, nearby
perturbations and support over the fractional grid. Finite grid checks are
regressions, not a proof over the continuous domain; the fixed-matrix affine-RHS
LP dual argument supplies that guarantee. The four binary stored examples were
not arithmetically wrong; they failed to expose a reusable-oracle defect.


In [6]:
zfraction = (5/60, 0.0)
rfraction = recourse(zfraction)
bad_beta = rfraction.value - rfraction.capacity_only * zfraction[1]
full_beta = rfraction.value - rfraction.grad[1] * zfraction[1]
@assert isapprox(rfraction.value, 2450; atol=1e-6)
@assert isapprox(rfraction.capacity_only, -30240; atol=1e-6)
@assert isapprox(bad_beta, 4970; atol=1e-6)
@assert isapprox(rfraction.grad[1], -15000; atol=1e-6)
@assert isapprox(full_beta, 3700; atol=1e-6)
for x in [0.0, 5/60-1e-5, 5/60, 5/60+1e-5, .2, .5, 1.0], y in [0.0, .25, 1.0]
    @assert full_beta + rfraction.grad[1]*x + rfraction.grad[2]*y <= recourse((x,y)).value + 1e-6
end
fd = (recourse((5/60+1e-5,0)).value - recourse((5/60-1e-5,0)).value)/2e-5
@assert isapprox(fd, -15000; atol=1e-4)
println((R=rfraction.value, capacity_only=rfraction.capacity_only,
         wrong_intercept=bad_beta, full_slope=rfraction.grad[1], full_intercept=full_beta,
         finite_difference=fd))
println("Every successful operate call asserts primal/dual feasibility status, primal residual and dual objective agreement.")


(R = 2450.0, capacity_only = -30240.0, wrong_intercept = 4970.0, full_slope = -15000.0, full_intercept = 3700.0, finite_difference = -15000.000000145517)
Every successful operate call asserts primal/dual feasibility status, primal residual and dual objective agreement.
